# Conectando el Hardware

**RESUMEN**: *En este notebook, aprenderás a configurar y conectar tu hardware ChipWhisperer. También explicaremos cómo compilar el firmware para tu microcontrolador objetivo, cómo capturar trazas de potencia y cómo comunicarte con los dispositivos objetivo.*

*Todas las llamadas a la API que utilizaremos aquí están documentadas en la página de [readthedocs de ChipWhisperer](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html), así que no dudes en abrirla en otra pestaña y seguirla también desde allí.*

**OBJETIVOS DE APRENDIZAJE:**
* Configurar el hardware de ChipWhisperer
* Utilizar la API de Python de ChipWhisperer para conectarse al hardware
* Comunicación con el dispositivo objetivo
* Captura de una traza de potencia

## Setup Físico

Para el desarrollo de estas prácticas utilizaremos la [ChipWhisperer-Nano (CW-Nano)](https://chipwhisperer.readthedocs.io/en/latest/Capture/ChipWhisperer-Nano.html), una plataforma de bajo coste diseñada específicamente para el aprendizaje y análisis de seguridad por canales laterales (*Side-Channel Analysis*). Fabricada por [NewAE Technology Inc.](https://www.newae.com/), esta tarjeta compacta integra en una sola placa tanto el osciloscopio de captura como el microcontrolador objetivo (target), lo que nos permitirá explorar de forma práctica ataques mediante análisis de consumo de potencia (*Power Analysis*) y ataques de inyección de fallos (*Fault Injection*).


### ChipWhisperer-Nano

El ChipWhisperer-Nano es un dispositivo de una sola placa. Incluye tanto el hardware de captura (atacante) junto con un objetivo STM32F0 integrado (víctima). Para usar este objetivo, simplemente conecta el dispositivo y listo:

<img src="img/cwnano.jpg" alt="CW-Nano Plugged In" width=400>

Para los notebooks de las prácticas deberás usar `PLATFORM='CWNANO'` ya que el software utilizado [ChipWhisperer](https://chipwhisperer.readthedocs.io/en/latest/) soporta diversas plataformas hardware.

## Conectando a ChipWhisperer-Nano

Ahora que el hardware está completamente configurado, podemos aprender a conectarnos a él. Podemos conectarnos al ChipWhisperer con:

In [ ]:
import chipwhisperer as cw
scope = cw.scope()

Por defecto, ChipWhisperer intentará detectar automáticamente el tipo de dispositivo que estás utilizando (CWLite/CW1200 o CWNano); consulta la documentación de la API para especificar manualmente el tipo de osciloscopio (scope). Si tienes varios dispositivos ChipWhisperer conectados, deberás indicar el número de serie del dispositivo al que deseas conectarte:

```python
scope = cw.scope(sn='<some long string of numbers>')
```

Para más información consulta la [sección API en readthedocs](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html).

Conectarse al dispositivo objetivo (víctima) también es sencillo:

In [ ]:
target = cw.target(scope, cw.targets.SimpleSerial) #cw.targets.SimpleSerial can be omitted

Solo trataremos el tipo de objetivo predeterminado, que es SimpleSerial. 

Se pueden definir algunos ajustes predeterminados mediante:

In [ ]:
scope.default_setup()

lo cual, según su [documentación](https://chipwhisperer.readthedocs.io/en/latest/api.html#chipwhisperer.scopes.cwnano.CWNano.default_setup), realiza lo siguiente para la CWNano:

* Configura el reloj del target (víctima) y del ADC (osciloscopio) a 10 MHz (reloj síncrono)
* Configura el osciloscopio para capturar 1000 muestras
* Establece el offset del osciloscopio en 0 (comenzará a capturar en cuanto se active el trigger)
* Configura el trigger del osciloscopio por flanco de subida (*rising edge*)
* Configura la velocidad de comunicación serie en 38400 baudios (8N1) para comunicarse con el target

¡Y listo! Tu ChipWhisperer ya está configurado y listo para atacar al objetivo.

**NOTA:** Necesitarás desconectar el osciloscopio/target antes de conectarte de nuevo, al igual que harías en otro notebook. Esto se puede hacer mediante `scope.dis()` y `target.dis()`.

## Compilación y carga del firmware

El siguiente paso para atacar un objetivo es compilar el firmware (código que se ejecutará en la víctima) y cargarlo en él. La mayor parte del firmware para la mayoría de los objetivos de ChipWhisperer se puede compilar usando nuestro sistema de compilación, siempre que tengas instalado el compilador adecuado (consulta https://chipwhisperer.readthedocs.io/en/latest/installation.html para más información sobre los compiladores).

El firmware debe compilarse desde la línea de comandos. Por suerte, gracias a Jupyter, podemos ejecutar un comando dentro de un cuaderno de la siguiente manera:

In [ ]:
%%bash
cd ../chipwhisperer/firmware/mcu/simpleserial-base/
make PLATFORM= CRYPTO_TARGET=NONE

Deberías ver una lista extensa de las plataformas (`PLATFORM`) para las que se puede compilar. Dejamos el parámetro `PLATFORM` en blanco en el comando anterior, por lo que el sistema de compilación imprimió, en su lugar, la lista de plataformas compatibles. Completa con nuestra plataforma `CWNANO`, vuelve a ejecutar el comando de compilación y el firmware debería compilarse con éxito.

Continuando con el proceso, ahora hay que cargar el firmware en tu objetivo (la víctima en la `CWNANO` es un `STM32`):

In [ ]:
cw.program_target(scope, cw.programmers.STM32FProgrammer, "../chipwhisperer/firmware/mcu/simpleserial-base/simpleserial-base-CWNANO.hex")

Si todo ha ido bien, el último mensaje indicará un `OK` para la verificación de la flash. 

Una vez cargado el firmware en el objetivo, a continuación, aprenderemos a capturar trazas de potencia y a comunicarnos con el dispositivo objetivo.

## Comunicación con el objetivo

La comunicación con los objetivos, que se realiza a través del objeto `SimpleSerial target` que obtuvimos anteriormente, se divide en dos categorías:

1. Comunicación serie directa (*raw serial*) mediante `target.read()`, `target.write()`, `target.flush()`, etc.

2. Comandos SimpleSerial mediante `target.simpleserial_read()`, `target.simpleserial_write()`, `target.simpleserial_wait_ack()`, etc.

El firmware que cargamos utiliza el protocolo SimpleSerial (https://wiki.newae.com/SimpleSerial), por lo que empezaremos con SimpleSerial. Más adelante, utilizaremos los comandos serie directos para enviar los mismos mensajes.

Si revisas el firmware base de SimpleSerial (`simpleserial-base.c`), verás que ante el comando SimpleSerial `'p'`, el objetivo devuelve como respuesta (*echo*) el mismo comando (el comando `'p'` espera un mensaje de 16 bytes a continuación). Vamos a probarlo ahora:

In [ ]:
msg = bytearray([0]*16) #simpleserial uses bytearrays
target.simpleserial_write('p', msg)

Comprobemos si obtuvimos una respuesta:

In [ ]:
print(target.simpleserial_read('r', 16))

Puedes cambiar el valor del `bytearray` de 0 a otro número para ver cómo cambia la respuesta.

También hay un comando `'k'` que simula la recepción de una clave (normalmente de 16 bytes). Prueba a enviarlo:

Ese comando no nos devuelve nada, pero debería enviar un acuse de recibo (*ack*) y devolvernos un código de error:

In [ ]:
print(target.simpleserial_wait_ack()) #should return 0

Los mensajes de SimpleSerial suelen tener la siguiente forma:

```python
caracter_de_comando + bytes_codificados_en_ascii + '\n'
```

Para nuestro primer comando, `command_character='p'` y `bytes_codificados_en_ascii="00"*16` (ten en cuenta que esto no es un `0x00` binario, es el texto ASCII `"00"`, cuyo valor binario es `0x3030`). Intenta reenviar el comando `'p'` anterior usando `target.write()`:

In [ ]:
target.write('p' + ) #fill in the rest here

En el protocolo SimpleSerial, la respuesta viene precedida por el carácter `r`, mientras que el código de confirmación/error (ACK) viene precedido por el carácter `z`.

Una simple llamada a `target.read()` devolverá todos los caracteres que el objetivo ha respondido hasta ahora. Veamos qué nos devolvió el dispositivo:

In [ ]:
recv_msg = ""

In [ ]:
recv_msg += target.read() #you might have to run this block a few times to get the full message
print(recv_msg)

Los comandos de SimpleSerial suelen ser suficientes para comunicarse con el firmware de SimpleSerial, pero necesitarás los comandos serie directos (*raw serial*) para algunas de las otras prácticas.

## SimpleSerial 2

A partir de ChipWhisperer 5.4, se encuentra disponible un nuevo protocolo de comunicación: SimpleSerial 2. Ofrece una serie de ventajas frente al protocolo SimpleSerial anterior:

1. Los datos son binarios sin codificar (excepto por el relleno de bytes o *byte stuffing*) en lugar de estar codificados en ASCII, lo que significa que se pueden enviar muchos más datos por paquete (casi el doble).
2. Dispone de campos de comando y subcomando que se pasan a las funciones de devolución de llamada (*callbacks*) del objetivo, lo que permite realizar más acciones por paquete. Por ejemplo, el firmware `simpleserial-aes` se ha modificado para permitir la configuración del texto plano, la clave y/o la máscara mediante un único paquete.
3. Cuenta con un campo de longitud, lo que permite que los mismos comandos del objetivo admitan paquetes de diferente longitud según la situación. Esto permite, por ejemplo, enviar tanto el texto plano como la clave (o solo uno de ellos) con el mismo comando en `simpleserial-aes`.
4. Incluye un CRC de 8 bits (0xA6) para asegurar la integridad de los datos.
5. Las tramas utilizan relleno de bytes con sobrecarga constante (COBS, *Consistent Overhead Byte Stuffing*). Esto facilita reiniciar la comunicación (lo cual se logra simplemente enviando dos bytes `0x00`) y ayuda a detectar bytes de longitud con formato incorrecto.

Sin embargo, se trata de un protocolo más complejo. Puedes utilizarlo compilando el firmware con `SS_VER=SS_VER_2_0` y usando `cw.targets.SimpleSerial2`.

## Captura de trazas

Ahora que el objetivo está programado y sabemos cómo comunicarnos con él, ¡comencemos a registrar algunas trazas de potencia! Para capturar una traza:

1. Arma el ChipWhisperer con `scope.arm()`. Comenzará a capturar tan pronto como se dispare el disparador (*trigger*) (que en nuestro caso es un flanco de subida en `gpio4`).
2. `scope.capture()` leerá la traza de potencia capturada, bloqueando la ejecución hasta que el ChipWhisperer termine de grabar o se agote el tiempo de espera (*timeout*) del osciloscopio. Ten en cuenta que el código de error devuelto te indicará si el osciloscopio agotó el tiempo de espera o no. No devuelve los datos capturados por el osciloscopio.
3. Puedes leer la traza de potencia capturada con `scope.get_last_trace()`.

`simpleserial_base` activará el disparador del ChipWhisperer cuando enviemos el comando `'p'`. Intenta capturar una traza ahora (puedes imprimirla con `print` o crear un gráfico con `matplotlib` como se ha visto en el notebook de introducción a Jupyter):

In [ ]:
scope.arm()
target.simpleserial_write('p', msg)
## fill in the rest...

ChipWhisperer también cuenta con una función práctica llamada `capture_trace()` que:

1. Opcionalmente envía el comando `'k'`
2. Arma el osciloscopio
3. Envía el comando `'p'`
4. Captura la traza
5. Lee el mensaje de respuesta `'r'`
6. Devuelve un objeto de clase `Trace` que agrupa los datos de la traza, el mensaje `'p'`, el mensaje `'r'` y el mensaje `'k'`

No siempre es la mejor opción a utilizar, pero suele ser suficiente para la mayoría de las aplicaciones con SimpleSerial.

## Conclusión

¡Y eso es todo! Ya deberías estar completamente listo para continuar con los siguientes notebooks.

Hemos pasado por alto algunos detalles aquí, así que consulta la [documentación de la API](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html) si te quedas atascado.

Como último paso, deberíamos desconectarnos del hardware para que este cuaderno no lo mantenga marcado como "en uso".

In [ ]:
scope.dis()
target.dis()

**Agradecimientos / Referencias:**  
Este material está basado y adaptado a partir del notebook original [*1 - Connecting to Hardware*](https://github.com/newaetech/chipwhisperer-jupyter/blob/e557a91b20e2a77c1f4675dff5d4138a20bf6f25/1%20-%20Connecting%20to%20Hardware.ipynb) del repositorio oficial de [NewAE Technology](https://github.com/newaetech/chipwhisperer-jupyter).